# 05 — Season-Level Data Integration & Full Player Universe

**CRISP-DM phase:** Data Preparation (revisited) / Modelling (extended).

**Why this notebook exists.** After the Task #4 evaluation (`outputs/results/task4_evaluation_findings.md`), it was discovered that FBref's SEASON-level aggregated tables have full 2017/18-2022/23 coverage across all five big-5 leagues -- unlike the match-level tables used in notebooks 02-04, which are unpopulated for Serie A/Ligue 1 before 2023/24 (`decision_log.md` D-006). This closes most of the De Zerbi/Sassuolo data gap: Sassuolo's entire three-season, whole-season tenure (SAFE per the tenure table) can now be sourced from real data instead of contributing nothing to 5 of 7 tactical dimensions. It also unlocks a genuine feature upgrade: season-level data through 2021/22 still carries the original StatsBomb `Pressures` count, a better pressing signal than the match-level pipeline's tackle-location proxy.

This notebook: (1) builds the full multi-league player universe from season-level data (needed for the app's player browser, independent of the three managers), (2) recomputes De Zerbi's manager profile with Sassuolo's data restored, (3) re-runs the compatibility and club-bias checks to see whether the improved data changes the Task #4 finding.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

from data_sources import DATA_PROCESSED
from football_features import (
    DIMENSION_WEIGHTS, build_concept_scores, build_dimension_scores,
    build_manager_profiles, build_club_within_manager_profiles, position_group,
)
from season_features import (
    build_season_league_dataset, build_season_population_baseline, build_manager_season_rows,
)
from compatibility import compute_compatibility, masked_cosine_similarity_matrix

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

dim_cols = list(DIMENSION_WEIGHTS.keys())

## 1. Build the full season-level league dataset and population baseline

In [2]:
season_df = build_season_league_dataset()
print(season_df.shape, "player-season rows, all big-5 leagues, 2017/18-2022/23")
season_baseline = build_season_population_baseline(season_df)
season_baseline.to_parquet(DATA_PROCESSED / "season_population_baseline.parquet", index=False)
print(season_baseline.shape)

(38041, 161) player-season rows, all big-5 leagues, 2017/18-2022/23
(210, 6)


## 2. Concept and dimension scores for the FULL player universe

Every player in the big-5 leagues, not just the three managers' clubs -- this is the foundation for the app's player browser and comparison plots (Task #7).

In [3]:
concept_scores_all, pca_diag_all = build_concept_scores(season_df, population_baseline=season_baseline)
dim_scores_all = build_dimension_scores(concept_scores_all)

universe = pd.concat([
    season_df[["Player", "Player_Href", "Squad", "Comp", "Season_End_Year", "Pos", "Min"]],
    concept_scores_all, dim_scores_all,
], axis=1)
universe = universe[universe.Min >= 450]  # at least ~5 full matches, avoids noisy small-sample profiles
print(universe.shape, "player-season rows with >=450 minutes")
universe.to_parquet(DATA_PROCESSED / "player_universe.parquet", index=False)
universe.sort_values("Min", ascending=False).head(10)

(11132, 24) player-season rows with >=450 minutes


,Player,Player_Href,Squad,Comp,Season_End_Year,Pos,Min,progressive_passing,progressive_carrying,possession_retention,chance_creation,box_presence,pressing_intensity,individual_defending,aerial_play,crossing,goalkeeper_distribution,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD9_pressing_intensity,TD11_individual_defending
15411,James Tarkowski,NaN,Burnley,Premier League,2020,DF,3420.0,-0.252940,-0.519295,-0.297809,1.027505,2.306100,-0.391818,0.149642,1.734217,-0.596936,NaN,-0.269766,-0.252940,-0.519295,0.540172,2.306100,-0.391818,0.625015
31543,Régis Gurtner,NaN,Amiens,Ligue 1,2019,GK,3420.0,-0.299164,-0.069590,-0.399655,-0.085158,-0.169352,-0.058944,-0.215249,-0.223709,-0.074411,-0.648188,-0.399116,-0.299164,-0.069590,-0.081934,-0.169352,-0.058944,-0.217787
1432,Alex Cordaz,NaN,Crotone,Serie A,2018,GK,3420.0,-0.251905,-0.069590,-1.011066,-0.079477,-0.169352,-0.100745,-0.134230,-0.223709,-0.074411,-0.879102,-0.605093,-0.251905,-0.069590,-0.077957,-0.169352,-0.100745,-0.161074
29364,Predrag Rajković,NaN,Reims,Ligue 1,2022,GK,3420.0,0.596699,-0.069590,0.172936,-0.103673,0.832938,0.099176,-0.215249,-0.867418,-0.074411,0.143183,0.378867,0.596699,-0.069590,-0.094895,0.832938,0.099176,-0.410900
9777,Emiliano Martínez,NaN,Aston Villa,Premier League,2021,GK,3420.0,0.104544,-0.069590,-0.028627,0.420541,-0.169352,-0.038603,-0.215249,-0.223709,-0.074411,-0.288225,-0.013961,0.104544,-0.069590,0.272055,-0.169352,-0.038603,-0.217787
19142,Kasper Schmeichel,NaN,Leicester City,Premier League,2021,GK,3420.0,-0.177161,-0.069590,-0.428394,-0.297072,-0.169352,-0.033396,-0.134230,-0.223709,-0.074411,0.013115,-0.214476,-0.177161,-0.069590,-0.230274,-0.169352,-0.033396,-0.161074
1391,Alessio Cragno,NaN,Cagliari,Serie A,2019,GK,3420.0,-0.009095,-0.069590,-0.065767,-0.297072,-0.169352,-0.048609,-0.215249,-0.223709,-0.074411,-0.487726,-0.121823,-0.009095,-0.069590,-0.230274,-0.169352,-0.048609,-0.217787
17442,Jordan Pickford,NaN,Everton,Premier League,2018,GK,3420.0,-0.021171,-0.069590,-0.231092,0.584731,0.787915,0.042770,-0.215249,0.372873,-0.074411,-0.848335,-0.249580,-0.021171,-0.069590,0.386988,0.787915,0.042770,-0.038813
3944,Baptiste Reynet,NaN,Nîmes,Ligue 1,2021,GK,3420.0,0.167995,-0.069590,0.590231,1.431522,1.454523,0.010091,-0.215249,0.372873,-0.074411,-0.011112,0.258845,0.167995,-0.069590,0.979742,1.454523,0.010091,-0.038813
21681,Luka Milivojević,NaN,Crystal Palace,Premier League,2019,MF,3420.0,-0.092079,-0.306902,-0.017980,0.405474,-0.459198,-0.067321,0.005100,0.387943,0.175395,NaN,-0.064292,-0.092079,-0.306902,0.336450,-0.459198,-0.067321,0.119953


## 3. Recover Sassuolo's manager-attributed profile from season-level data

In [4]:
sassuolo_rows = build_manager_season_rows(season_df)
sassuolo_concepts, sassuolo_pca = build_concept_scores(sassuolo_rows, population_baseline=season_baseline)
sassuolo_dims = build_dimension_scores(sassuolo_concepts)

sassuolo_profiles = pd.concat([
    sassuolo_rows[["Player", "Player_Href", "manager", "team", "Season_End_Year", "Pos", "Min"]].rename(columns={"Min": "total_minutes"}),
    sassuolo_concepts, sassuolo_dims,
], axis=1)
print(sassuolo_profiles.shape)
print("\nDimensions now populated for Sassuolo (previously all NaN except pressing/individual_defending):")
sassuolo_profiles[dim_cols].describe().T[["count", "mean"]]

(337, 24)

Dimensions now populated for Sassuolo (previously all NaN except pressing/individual_defending):


,count,mean
TD1_buildup_directness,335.0,0.453420
TD3_progression_passing,335.0,0.245345
TD4_progression_carrying,335.0,0.048377
TD6_chance_creation,336.0,-0.101372
TD7_box_presence,336.0,0.002064
TD9_pressing_intensity,332.0,0.471472
TD11_individual_defending,336.0,-0.114367


## 4. Merge into the existing match-level player profiles, replacing Sassuolo's rows

In [5]:
player_profiles_v1 = pd.read_parquet(DATA_PROCESSED / "player_profiles.parquet")
before = player_profiles_v1[player_profiles_v1.team != "Sassuolo"]
player_profiles_v2 = pd.concat([before, sassuolo_profiles], ignore_index=True)
print(f"v1: {len(player_profiles_v1)} rows -> v2: {len(player_profiles_v2)} rows ({len(sassuolo_profiles)} Sassuolo rows now season-level-sourced)")
player_profiles_v2.to_parquet(DATA_PROCESSED / "player_profiles.parquet", index=False)

v1: 2711 rows -> v2: 2630 rows (337 Sassuolo rows now season-level-sourced)


## 5. Recompute manager profiles and re-run the club-bias / compatibility checks

In [6]:
manager_profiles_v2 = build_manager_profiles(player_profiles_v2, dim_cols)
manager_profiles_v2.to_parquet(DATA_PROCESSED / "manager_profiles.parquet", index=False)
manager_profiles_v2.set_index("manager").round(3)

,TD1_buildup_directness,TD3_progression_passing,TD4_progression_carrying,TD6_chance_creation,TD7_box_presence,TD9_pressing_intensity,TD11_individual_defending
manager,,,,,,,
Jürgen Klopp,0.318,0.237,-0.012,-0.035,0.079,0.229,-0.139
Pep Guardiola,0.446,0.293,0.102,-0.037,0.056,0.304,-0.182
Roberto De Zerbi,0.251,0.139,0.015,-0.017,-0.022,-0.001,-0.163


In [7]:
club_profiles_v2 = build_club_within_manager_profiles(player_profiles_v2, dim_cols)
labels = club_profiles_v2["manager"] + " @ " + club_profiles_v2["team"]
vecs = club_profiles_v2[dim_cols].to_numpy(dtype=float)
sim, ndims = masked_cosine_similarity_matrix(vecs)
sim_matrix_v2 = pd.DataFrame(sim, index=labels, columns=labels)
ndims_matrix_v2 = pd.DataFrame(ndims, index=labels, columns=labels)
print("Similarity (v2, Sassuolo now real data):")
display(sim_matrix_v2.round(3))
print("\nDimensions used per pair:")
display(ndims_matrix_v2)

Similarity (v2, Sassuolo now real data):


,Jürgen Klopp @ Liverpool,Pep Guardiola @ Manchester City,Roberto De Zerbi @ Benevento,Roberto De Zerbi @ Brighton & Hove Albion,Roberto De Zerbi @ Marseille,Roberto De Zerbi @ Sassuolo
Jürgen Klopp @ Liverpool,1.000,0.979,-0.017,0.917,0.773,0.764
Pep Guardiola @ Manchester City,0.979,1.000,-0.022,0.887,0.757,0.801
Roberto De Zerbi @ Benevento,-0.017,-0.022,1.000,0.626,-0.026,0.843
Roberto De Zerbi @ Brighton & Hove Albion,0.917,0.887,0.626,1.000,0.830,0.823
Roberto De Zerbi @ Marseille,0.773,0.757,-0.026,0.830,1.000,0.743
Roberto De Zerbi @ Sassuolo,0.764,0.801,0.843,0.823,0.743,1.000



Dimensions used per pair:


,Jürgen Klopp @ Liverpool,Pep Guardiola @ Manchester City,Roberto De Zerbi @ Benevento,Roberto De Zerbi @ Brighton & Hove Albion,Roberto De Zerbi @ Marseille,Roberto De Zerbi @ Sassuolo
Jürgen Klopp @ Liverpool,7,7,2,7,7,7
Pep Guardiola @ Manchester City,7,7,2,7,7,7
Roberto De Zerbi @ Benevento,2,2,2,2,2,2
Roberto De Zerbi @ Brighton & Hove Albion,7,7,2,7,7,7
Roberto De Zerbi @ Marseille,7,7,2,7,7,7
Roberto De Zerbi @ Sassuolo,7,7,2,7,7,7


In [8]:
dz_rows = [i for i, l in enumerate(labels) if l.startswith("Roberto De Zerbi")]
other_rows = [i for i in range(len(labels)) if i not in dz_rows]

def well_covered_pairs(rows_a, rows_b, min_dims=4):
    return [sim[i, j] for i in rows_a for j in rows_b if i != j and ndims[i, j] >= min_dims]

within_dz = well_covered_pairs(dz_rows, dz_rows)
dz_to_others = well_covered_pairs(dz_rows, other_rows)
print(f"De Zerbi club-to-club (v2): mean={np.mean(within_dz):.3f} (n={len(within_dz)})")
print(f"De Zerbi vs others (v2):    mean={np.mean(dz_to_others):.3f} (n={len(dz_to_others)})")

De Zerbi club-to-club (v2): mean=0.799 (n=6)
De Zerbi vs others (v2):    mean=0.816 (n=6)


In [9]:
compat_v2 = compute_compatibility(player_profiles_v2, manager_profiles_v2, dim_cols, min_dims_required=4)
confident_v2 = compat_v2[~compat_v2.low_confidence]
best_fit_v2 = confident_v2.loc[confident_v2.groupby(["Player", "Season_End_Year", "team"])["compatibility_similarity"].idxmax()]
match_rate_v2 = (best_fit_v2["scored_against_manager"] == best_fit_v2["actual_manager"]).mean()
print(f"v2 best-fit match rate: {match_rate_v2:.1%} (n={len(best_fit_v2)}), vs v1's 32.8% (n=470)")
pd.crosstab(best_fit_v2["actual_manager"], best_fit_v2["scored_against_manager"])

v2 best-fit match rate: 42.1% (n=561), vs v1's 32.8% (n=470)


scored_against_manager,Jürgen Klopp,Pep Guardiola,Roberto De Zerbi
actual_manager,,,
Jürgen Klopp,66,44,78
Pep Guardiola,46,47,77
Roberto De Zerbi,34,46,123


In [10]:
compat_v2.to_parquet(DATA_PROCESSED / "player_manager_compatibility.parquet", index=False)
sim_matrix_v2.to_csv(Path.cwd().parent / "outputs" / "tables" / "club_bias_similarity_matrix_v2.csv")
print("saved v2 outputs")

saved v2 outputs
